# CNN for 52 Card + 1 Joker deck classification

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import cv2
import matplotlib.pyplot as plt

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device is: {device}")

Device is: cuda


In [42]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.ImageFolder(root='../card-dataset/train', transform=transform)
test_dataset = datasets.ImageFolder(root='../card-dataset/test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Definición de la Red

In [48]:
import torch.nn as nn
import torch.nn.functional as F

class CardNet(nn.Module):
    def __init__(self):
        super(CardNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, stride=1, padding=2)

        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2)

        self.fc1 = nn.Linear(64 * 55 * 55, 120)
        self.fc2 = nn.Linear(120, 53)

    def forward(self, x):
        print(x.shape)
        x = self.pool(F.relu(self.conv1(x)))
        print(x.shape)
        x = self.pool(F.relu(self.conv2(x)))
        print(x.shape)
        x = x.view(-1, 64 * 55 * 55)
        print(x.shape)
        x = F.relu(self.fc1(x))
        print(x.shape)
        x = self.fc2(x)
        return x

myModel = CardNet()
img, label = train_dataset[0]
cuda_myModel = myModel.to(device)
myModel.to(device).forward(img.to(device))


Label: 0
torch.Size([3, 224, 224])
torch.Size([32, 111, 111])
torch.Size([64, 55, 55])
torch.Size([1, 193600])
torch.Size([1, 120])


tensor([[ 0.0224,  0.1820, -0.0821,  0.0133,  0.0279, -0.0287,  0.0237, -0.0904,
         -0.0372,  0.0463,  0.0149, -0.1102,  0.0239, -0.0338,  0.0608,  0.1186,
          0.0187, -0.0765, -0.0351,  0.0937, -0.0053,  0.0609, -0.1507, -0.0283,
          0.1519,  0.0312,  0.0986, -0.0251, -0.0416,  0.1171,  0.1492, -0.0004,
         -0.0540, -0.0199, -0.0772, -0.1062, -0.0808,  0.0541, -0.0547,  0.0979,
         -0.0249, -0.0311, -0.0764, -0.0553,  0.0010, -0.1754,  0.0019,  0.0256,
          0.0657,  0.1248,  0.0267, -0.0179, -0.0305]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

# Bucle de entrenamiento

In [ ]:
import torch.optim as optim

model = CardNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Entrenamiento
EPOCHS = 10
for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()  # Limpiamos los gradientes
        outputs = model(images)  # Pasamos las imágenes por la red
        loss = criterion(outputs, labels)  # Calculamos la pérdida
        loss.backward()  # Backpropagation
        optimizer.step()  # Actualizamos los pesos
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')